# Semana 3 · Sesión 1: Git en tu máquina

**Módulo 0**

## Objetivos de la sesión

1. Explicar el modelo de tres áreas de Git y registrar cambios con `init`,
   `add`, `commit` y `status`.
2. Leer la historia de un repositorio con `log` y `diff`.
3. Crear ramas, fusionarlas y resolver un conflicto sin entrar en pánico.


## Antes de empezar

Revisa el checklist de [`preparacion.md`](../preparacion/preparacion.md):
hoy damos por hecho que `git --version` responde y que ya configuraste tu
nombre y tu correo.

Esta sesión es distinta a las anteriores en una cosa importante: **la
herramienta del día es la terminal, no Python**. Los comandos de Git
aparecen en bloques como este

```bash
git status
```

y esos bloques **se teclean en la terminal**, no son celdas del notebook.
Ten las dos ventanas lado a lado: el notebook como guion y bitácora, la
terminal para trabajar. Solo hay dos celdas de código en toda la sesión, al
final, cuando veamos cómo se ve por dentro un archivo en conflicto.


## ¿Por qué control de versiones?

Todos hemos tenido esta carpeta:

```
tarea_final.ipynb
tarea_final_v2.ipynb
tarea_final_v2_corregida.ipynb
tarea_final_v3_buena_ESTA_SI.ipynb
```

Y todos hemos hecho las tres preguntas que esa carpeta no puede responder:
¿qué cambió entre la v2 y la v3?, ¿por qué lo cambié?, ¿cómo vuelvo a la
versión de antes de romperlo?

Un sistema de control de versiones responde las tres. Guarda una serie de
**instantáneas** del proyecto completo, cada una con un mensaje que dice
qué cambió y por qué, y te deja volver a cualquiera de ellas. Además —y por
eso lo usa todo el software del mundo, incluido SymPy— permite que varias
personas trabajen sobre los mismos archivos sin pisarse.

Git es ese sistema. GitHub, que veremos en la sesión 2, es un sitio donde
poner una copia de tu repositorio para compartirla; no son lo mismo.


## Las tres áreas

Este es el modelo mental que hay que tener claro antes de teclear nada. Un
repositorio de Git tiene tres áreas, y cada comando mueve tus cambios de
una a la siguiente:

![Las tres áreas de Git y los comandos que mueven cambios entre ellas](img/git-tres-areas.svg)

| Área | Qué contiene | Cómo llegan las cosas ahí |
|---|---|---|
| Directorio de trabajo | Los archivos como los ves en tu carpeta | Editándolos |
| Área de preparación (*staging*) | Los cambios que van a entrar en el próximo commit | `git add` |
| Historial local (`.git`) | Todas las instantáneas ya guardadas | `git commit` |
| Repositorio remoto | La copia compartida, en GitHub | `git push` |

El área de preparación es la que suele confundir, porque parece un trámite
de más. Sirve para **elegir qué entra en cada commit**: si tocaste cinco
archivos pero solo dos corresponden al arreglo que quieres registrar,
agregas esos dos y dejas los otros para un commit aparte.

Todo el historial vive en una carpeta oculta llamada `.git`, en la raíz de
tu proyecto. Si la borras, borras el historial completo; el resto de tus
archivos no tienen nada de especial.


## Empezar un repositorio y guardar el primer cambio

```bash
mkdir practica-git          # una carpeta cualquiera
cd practica-git
git init                    # crea .git: a partir de aquí es un repositorio

git status                  # ¿en qué estado están mis archivos?
```

`git status` es el comando que más vas a usar en tu vida. Contesta las dos
preguntas del diagrama de arriba: qué cambió y no está preparado, y qué
está preparado y no se ha guardado. Cuando no sepas qué hacer, córrelo.

```bash
git add constantes.txt      # del directorio de trabajo al área de preparación
git commit -m "Agrega las constantes fundamentales del problema"
```

Sobre el mensaje: es lo que vas a leer dentro de seis meses cuando busques
cuándo se rompió algo. La convención de este curso —y la de casi todos los
proyectos— es escribirlo en **modo imperativo** y en una línea corta:
*"Agrega el cálculo de la energía"*, no *"agregué unas cosas"*. Los
mensajes de este repositorio siguen esa regla: cuando clones el curso,
córrele un `git log --oneline` y míralos.


## TODO en clase 1

Crea tu repositorio de práctica y hazle dos commits. Completa los huecos
(`____`) en tu terminal:

```bash
mkdir practica-git
cd practica-git
git ____                                  # inicia el repositorio

echo "g = 9.81" > constantes.txt          # crea el archivo (o usa tu editor)

git ____                                  # revisa en qué estado estás
git ____ constantes.txt                   # prepáralo
git ____ -m "____"                        # guárdalo, con mensaje en imperativo

echo "c = 299792458" >> constantes.txt    # agrega una segunda línea

git ____ constantes.txt
git ____ -m "____"                        # segundo commit

git log --oneline                         # deberías ver tus dos commits
```

Al terminar, corre `git status` una vez más: debe decir que no hay nada
pendiente. Ese es el estado limpio al que hay que volver siempre.


## Leer la historia: `log`, `diff` y `show`

```bash
git log --oneline              # una línea por commit: identificador y mensaje
git log --oneline --graph      # igual, pero dibujando las ramas
git show <identificador>       # qué cambió exactamente en ese commit
```

Y para ver lo que todavía no has guardado:

| Comando | Compara | Responde |
|---|---|---|
| `git diff` | Directorio de trabajo ↔ área de preparación | ¿Qué edité y aún no preparo? |
| `git diff --staged` | Área de preparación ↔ último commit | ¿Qué está a punto de entrar al commit? |
| `git status` | Las dos anteriores, en resumen | ¿Dónde estoy parado? |

Cada commit tiene un identificador largo (un *hash* como `d3148f4a...`),
pero casi siempre basta con los primeros siete caracteres.


## Depuración en vivo: `git diff` no muestra nada

El tropiezo número uno de la primera semana con Git:

```bash
git add constantes.txt
git diff                # ...no imprime nada. ¿Se perdió mi cambio?
```

No se perdió nada. Mira otra vez la tabla de arriba: `git diff` compara el
directorio de trabajo contra el **área de preparación**, y acabas de hacer
que sean idénticos con `git add`. Tu cambio está un casillero más adelante:

```bash
git diff --staged       # aquí sí aparece
```

La moraleja sirve para casi todo lo que se siente raro en Git: la pregunta
correcta no es "¿dónde está mi cambio?", sino "¿en cuál de las tres áreas
está?".


## Ramas: trabajar sin miedo a romper

Una **rama** es un nombre que apunta a un commit y que avanza contigo
conforme haces commits nuevos. Nada más. Por eso crear una es instantáneo y
gratis: no copia archivos, solo pone otro nombre en el mismo historial.

Sirven para trabajar en algo sin tocar la versión que ya funciona. Este
curso las usa así, y lo van a ver en el historial del repositorio:
`draft/semana-03` para preparar una clase, `fix/semana-02-typo` para
corregir una errata.

```bash
git switch -c unidades      # crea la rama "unidades" y se cambia a ella
git switch main             # regresa a main
git branch                  # lista las ramas; la actual lleva un asterisco
```

Los commits que hagas en `unidades` no existen en `main` hasta que las
fusiones.


## Fusionar: `git merge`

Fusionar es traer los commits de otra rama a la rama en la que estás
parado. Primero te paras en la rama de destino, y luego traes:

```bash
git switch main             # me paro donde quiero que lleguen los cambios
git merge unidades          # traigo los commits de "unidades"
```

Pueden pasar dos cosas:

- **Avance rápido** (*fast-forward*): si `main` no cambió desde que te
  ramificaste, Git simplemente mueve el nombre `main` hacia adelante. No
  hay commit nuevo.
- **Commit de fusión**: si las dos ramas avanzaron por su lado, Git crea un
  commit extra que tiene dos padres y junta ambas historias. Es lo que
  verás en el repositorio del curso como *"Merge pull request #8..."*.

Si las dos ramas tocaron **líneas distintas**, Git resuelve la fusión solo.
El caso interesante es el otro.


## TODO en clase 2

En tu repositorio de práctica, haz un cambio en una rama y fusiónalo:

```bash
git switch -c ____                        # crea una rama llamada "unidades"

echo "unidades = 'SI'" >> constantes.txt
git add constantes.txt
git ____ -m "____"                        # commit dentro de la rama

git ____ main                             # regresa a main
cat constantes.txt                        # la línea nueva NO está aquí

git ____ unidades                         # fusiona
cat constantes.txt                        # ahora sí

git log --oneline --graph                 # mira la forma de la historia
```

¿Fue avance rápido o commit de fusión? Compruébalo en el `log`: si no
aparece un commit con la palabra *"Merge"*, fue avance rápido.


## Cuando dos cambios chocan: conflictos

Un **conflicto** ocurre cuando las dos ramas modificaron **la misma línea**
del mismo archivo. Git no puede saber cuál de las dos versiones quieres, y
—esto es lo importante— no inventa: se detiene, te avisa cuáles archivos
están en conflicto y escribe **las dos versiones dentro del archivo**,
separadas por marcadores, para que tú decidas.

La celda de abajo muestra cómo se ve ese archivo por dentro. No es magia ni
un formato secreto: es texto que Git escribió ahí.


In [ ]:
# Así queda un archivo cuando Git no puede decidir por ti.
constantes_en_conflicto = """g = 9.81
<<<<<<< HEAD
c = 299792458          # la versión de la rama en la que estás parado
=======
c = 3.0e8              # la versión que llega de la otra rama
>>>>>>> unidades
"""

print(constantes_en_conflicto)

Los tres marcadores se leen así:

- `<<<<<<< HEAD` — a partir de aquí, **tu** versión (la de la rama actual).
- `=======` — la frontera entre las dos versiones.
- `>>>>>>> unidades` — hasta aquí, la versión que venía de la otra rama.

Resolver el conflicto es editar el archivo a mano hasta que diga lo que
debe decir —quedándote con una versión, con la otra, o con una mezcla— y
**borrar los tres marcadores**. Después:

```bash
git status                  # te dice qué archivos siguen en conflicto
git add constantes.txt      # "ya lo resolví"
git commit                  # cierra la fusión
```

Y si prefieres deshacer todo y volver a antes de la fusión:

```bash
git merge --abort
```


## Depuración en vivo: qué pasa si dejas los marcadores

Los marcadores no son comentarios: son texto que rompe el archivo. Si se te
cuela un `<<<<<<<` en un `.py` y haces commit, lo que subes ya no es código
válido — y en un `.ipynb` ni siquiera es JSON válido, así que el notebook
deja de abrir.

La celda de abajo lo comprueba: le pedimos a Python que compile el texto en
conflicto de la celda anterior.


In [ ]:
try:
    compile(constantes_en_conflicto, "constantes.py", "exec")
except SyntaxError as error:
    print("Con los marcadores dentro ya no es código válido:")
    print(" ", error)

## TODO en clase 3

Provoca un conflicto **a propósito** y resuélvelo. Es el ejercicio más
importante del día: la primera vez que veas uno, que sea aquí y no la noche
de la entrega.

```bash
git switch -c ____                        # crea la rama "conflicto"
# edita constantes.txt y cambia el valor de g a 9.8
git ____ constantes.txt
git ____ -m "Redondea la aceleración de la gravedad"

git switch main
# edita constantes.txt y cambia el valor de g a 9.80665, en la MISMA línea
git ____ constantes.txt
git ____ -m "Usa el valor estándar de la gravedad"

git ____ conflicto                        # fusiona: aquí explota

git ____                                  # ¿qué archivos están en conflicto?
# abre constantes.txt, decide qué valor se queda y borra los tres marcadores
git ____ constantes.txt                   # marca el conflicto como resuelto
git ____ -m "Resuelve el conflicto en el valor de g"
```

Cuando `git status` vuelva a decir que no hay nada pendiente, terminaste.


## Marcha atrás: cuando algo sale mal

Guarda esta tabla. Casi todo en Git se puede deshacer, y saberlo es lo que
quita el miedo a experimentar:

| Situación | Salida |
|---|---|
| Edité un archivo y quiero descartar el cambio | `git restore <archivo>` |
| Hice `git add` y quiero sacarlo del área de preparación | `git restore --staged <archivo>` |
| Me equivoqué en el mensaje del último commit | `git commit --amend -m "Mensaje nuevo"` |
| La fusión se complicó y quiero volver atrás | `git merge --abort` |
| Me perdí | `git status` y `git log --oneline --graph` |

Lo único que no se deshace es borrar la carpeta `.git`. Todo lo demás que
ya esté en un commit se puede recuperar.


## Resumen

Hoy trabajamos Git entero en tu máquina, sin internet de por medio: las
tres áreas y los comandos que mueven cambios entre ellas (`add`, `commit`),
cómo leer la historia (`status`, `log`, `diff`, `show`), y cómo trabajar en
paralelo con ramas, fusiones y conflictos resueltos a mano.

La idea que conviene llevarse: casi toda confusión con Git se aclara
preguntando *¿en cuál de las tres áreas está mi cambio?*

**Próxima sesión — Semana 3, sesión 2:** GitHub. Vamos a publicar tu
repositorio, hacer tu fork del curso y abrir tu primer Pull Request —que es
el entregable de la semana— y a ver por qué los notebooks se llevan tan mal
con Git y cómo se arregla.
